In [0]:
%pip install openai requests
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
import logging
from openai import OpenAI
import requests
import json
import os
from datetime import datetime
logging.basicConfig(level = logging.INFO, format='%(levelname)s: %(message)s')

In [0]:
OPENAI_API_KEY = dbutils.secrets.get(scope="anomaly_proj_secrets", key="openai_api_key")
DISCORD_URL = dbutils.secrets.get(scope="anomaly_proj_secrets", key="discord_webhook_url")


INFO: Received command c on object id p0


In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [0]:
dbutils.widgets.text("catalog", "real-time-streaming-lakehouse")
dbutils.widgets.text("schema", "ecommerce-events")

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

In [0]:
GOLD_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`gold-kpi-metrics`"
SIVER_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`silver-events`"
ALERT_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`alert-history`"

In [0]:
alert_df = spark.read.table(ALERT_TABLE)

In [0]:
gold_df = spark.read.table(GOLD_TABLE)
unalerted_df =( gold_df.join(
    alert_df,
    on=["window_start", "window_end"],
    how="left_anti"
   )
   .orderBy(F.col("window_start").desc())
   
)
rows = unalerted_df.limit(1).collect()
if not rows:
    logging.warning("No new unalerted gold window found")
    dbutils.notebook.exit("No new unalerted gold window found")
current = rows[0].asDict()
current

{'window_start': datetime.datetime(2026, 6, 22, 15, 15),
 'window_end': datetime.datetime(2026, 6, 22, 15, 20),
 'total_events': 48,
 'unique_users': 49,
 'total_revenue': 307.09,
 'purchase_count': 13,
 'conversion_rate': 0.2708333333333333,
 'avg_order_value': 23.62230769230769}

In [0]:
previous_df = (gold_df.filter(
    F.col("window_start")<current["window_start"]
).orderBy(F.col("window_start").desc())
)
rows = previous_df.limit(1).collect()
if not rows:
    logging.warning("No previous gold window found")
    dbutils.notebook.exit("No previous gold window found")
previous = rows[0].asDict()
previous

{'total_events': 166,
 'unique_users': 162,
 'total_revenue': 1006.03,
 'purchase_count': 38,
 'conversion_rate': 0.2289156626506024,
 'avg_order_value': 26.474473684210526,
 'window_start': datetime.datetime(2026, 6, 22, 15, 10),
 'window_end': datetime.datetime(2026, 6, 22, 15, 15)}

In [0]:
def detect_anomalies(current, previous):
    anomalies= []

    current_users = int(current["unique_users"] or 0)
    previous_users = int(previous["unique_users"] or 0)
    current_revenue = float(current["total_revenue"] or 0)
    previous_revenue = float(previous["total_revenue"] or 0)
    current_conversion = float(current["conversion_rate"] or 0)
    curr_purchase_count = int(current["purchase_count"] or 0)
    curr_total_events = int(current["total_events"] or 0)

    if current_conversion<0.01 and curr_total_events>50:
        anomalies.append("Conversion rate below 1%")
    if previous_revenue>0 and current_revenue<previous_revenue*0.6:
        anomalies.append("Revenue dropped more than 40% compared to previous window")
    if previous_users>0 and current_users<previous_users*0.5:
        anomalies.append("Unique users dropped more than 50% compared to previous window")
    if curr_purchase_count == 0 and curr_total_events>50:
        anomalies.append("No purchases detected despite meaningful traffic")
    return anomalies


In [0]:
anomalies = detect_anomalies(current, previous)
if not anomalies:
    schema = StructType([
        StructField("window_start", TimestampType(), True),
        StructField("window_end", TimestampType(), True),
        StructField("alert_send_at", TimestampType(), True),
        StructField("alert_reasons", ArrayType(StringType()), True),
        StructField("alert_text", StringType(), True)
    ])
    df = spark.createDataFrame([{
       "window_start":current["window_start"],
       "window_end":current["window_end"],
       "alert_send_at":None,
       "alert_reasons":[],
       "alert_text" :"No anomaly detected"
    }], schema=schema)
    df.write.mode("append").saveAsTable(ALERT_TABLE)
    logging.warning("No anomaly detected")
    dbutils.notebook.exit("No anomaly detected")

In [0]:
silver_df = spark.read.table(SIVER_TABLE)
window_events = silver_df.filter((F.col("event_timestamp")>= current["window_start"]) & (F.col("event_timestamp")<current["window_end"]))
window_events.show(truncate=False)
window_events.count()

+------+--------------------------+----------+--------+----------------+----------+------------------------------------+------------------------------------+-------------+-----------------------+-------------------------------------------------------------------------------------------+-----------------------+
|amount|event_timestamp           |event_type|page    |product_category|product_id|session_id                          |user_id                             |_rescued_data|ingested_at            |source                                                                                     |processed_at           |
+------+--------------------------+----------+--------+----------------+----------+------------------------------------+------------------------------------+-------------+-----------------------+-------------------------------------------------------------------------------------------+-----------------------+
|0.0   |2026-06-22 15:15:55.612855|click_nav |product |books    

48

In [0]:
def get_funnel_context():
    return (
        window_events.groupBy("event_type")
                 .agg(F.count("*").alias("event_count"))
                 .orderBy(F.col("event_count").desc())
                 .toPandas()
                 .to_string(index=False)
    )

def get_category_context():
    return (
        window_events.filter(F.col("event_type") == "purchase")
               .groupBy("product_category")
               .agg(
                   F.count("*").alias("purchase_count"),
                   F.sum("amount").alias("total_revenue")
               )
               .orderBy(F.col("total_revenue").desc())
               .toPandas()
               .to_string(index=False)
    )

def get_page_context():
    return (
        window_events.groupBy("page")
                     .agg(F.count("*").alias("page_events"))
                     .orderBy(F.col("page_events").desc())
                     .toPandas()
                     .to_string(index=False)
    )

In [0]:
funnel_context = get_funnel_context()
category_context = get_category_context()
page_context = get_page_context()

In [0]:
prompt = f"""
You are an ecommerce incident analyst. Do not claim certainty, use the below information to explain observations, hypotheses, and recommended investigations 
Current 5 minute KPI window:
- Window: {current["window_start"]} to {current["window_end"]}
- Total events: {current["total_events"]}
- Unique users: {current["unique_users"]}
- Total revenue: {current["total_revenue"]}
- Purchase count: {current["purchase_count"]}
- Conversion rate: {current["conversion_rate"]}
- Average order value: {current["avg_order_value"]}
Previous 5 minute KPI winow:
- Window: {previous["window_start"]} to {previous["window_end"]}
- Unique users: {previous["unique_users"]}
- Total revenue: {previous["total_revenue"]}
- Conversion rate: {previous["conversion_rate"]}
- Purchase count: {previous["purchase_count"]}

Detected anomalies:
{anomalies}
Supporting context
Funnel context:
{funnel_context}
Category context:
{category_context}
Page context:
{page_context}

Write a Discord incident alert under 140 words with only when anomaly is detected
1. What changed
2. What context suggests
3. Recommended Immediate checks

"""

In [0]:
response = client.responses.create(
    model="gpt-5-mini",
    input=prompt
)
alert_text = response.output_text
print(alert_text)

In [0]:
anomalies

In [0]:
discord_payload={
    'content':f"""
    **Anomaly Incident Alert**

    **Window:** {current["window_start"]} to {current["window_end"]}

    **Rules Trigerred:**
    {chr(10).join([f"-{a}" for a in anomalies])}

    **AI Analyst Summary:**
    {alert_text}
    """ 
}
if anomalies:
   result = requests.post(DISCORD_WEBHOOK_URL,json=discord_payload)
   print("Discord alert sent")
if result.status_code not in [200, 204]:
    raise Exception(f"Discord alert failed:{result.status_code}, {result.text}")


Discord alert sent


In [0]:
schema = StructType([
        StructField("window_start", TimestampType(), True),
        StructField("window_end", TimestampType(), True),
        StructField("alert_send_at", TimestampType(), True),
        StructField("alert_reasons", ArrayType(StringType()), True),
        StructField("alert_text", StringType(), True)
    ])
if anomalies:
    alert_record = spark.createDataFrame([{
        "window_start":current["window_start"],
        "window_end":current["window_end"],
        "alert_send_at":datetime.now(),
        "alert_reasons":anomalies,
        "alert_text" :alert_text
    }], schema=schema)
    alert_record.write.mode("append").saveAsTable(ALERT_TABLE)
    print("Alert History is updated")


Alert History is updated


In [0]:
%sql
Select * from `real-time-streaming-lakehouse`.`ecommerce-events`.`alert-history`

window_start,window_end,alert_send_at,alert_reasons,alert_text
2026-06-22T15:15:00.000Z,2026-06-22T15:20:00.000Z,2026-06-22T15:29:21.664Z,"List(Revenue dropped more than 40% compared to previous window, Unique users dropped more than 50% compared to previous window)","Anomaly: revenue ↓ ~69% (1006.03 → 307.09) and unique users ↓ ~70% (162 → 49) in 15:15–15:20. 1) What changed - Sharp drop in users and revenue; purchases fell 38 → 13 but conversion rate rose slightly (0.229 → 0.271), AOV ≈ $23.6. 2) What context suggests - Checkout/cart activity still present (checkout 16, cart 12). High logouts (17) and possible session churn. Could be traffic source outage, campaign/ingest change, analytics/tracking loss, or auth/session/payment failures causing user drop rather than pure conversion loss. 3) Recommended immediate checks - Traffic sources/ads and CDN/edge logs for drops or errors. - Analytics/tracking tag health (beacon loss). - Auth/session service & logout spike. - API/checkout/payment gateway 4xx/5xx logs and rate limiting. - Recent deploys/feature flags/A-B tests. - Inspect a few raw sessions/replays and load balancer/access logs."
2026-06-22T15:10:00.000Z,2026-06-22T15:15:00.000Z,null,List(),No anomaly detected
2026-06-18T22:15:00.000Z,2026-06-18T22:20:00.000Z,null,List(),No anomaly detected
2026-06-18T22:20:00.000Z,2026-06-18T22:25:00.000Z,null,List(),No anomaly detected
